# Variant Analysis — Prototyping Notebook

Prototype for SNP/indel detection from ONT reads mapped to **BTx623** (NC_012870.2 contig naming).

**Tool chain:**  `samtools index` → `Clair3 (run_clair3.sh)` → `bcftools filter` → `bcftools merge`

**Samples:** SBC4, SBC10, SBC11, SBC23

**Final Snakemake workflow:** `workflows/variant_analysis/Snakefile`

In [ ]:
import subprocess
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
WDIR        = Path("/home/daffa/Work/2026/02-JSPP67")
BAM_DIR     = Path("/home/daffa/hdd/2026/02-JSPP67_hdd/Sbi-TAA")
REF         = WDIR / "data/reference/asm_BTx623.fna"
RESULTS_DIR = WDIR / "results/variant_analysis"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES = ["SBC4", "SBC10", "SBC11", "SBC23"]
BAMS    = {s: BAM_DIR / f"read_{s}.bam" for s in SAMPLES}

# Clair3 model — update this to match your basecalling model
# Download models from: https://github.com/HKU-BAL/Clair3  or  rerio
CLAIR3_MODEL = "/path/to/clair3_models/r1041_e82_400bps_sup_v430"

print("Reference :", REF)
print("BAMs found:", [str(b) for b in BAMS.values() if b.exists()])
print("Missing   :", [str(b) for b in BAMS.values() if not b.exists()])

## Step 1 — Inspect BAM headers and index BAMs

Confirm that all four BAMs exist, are sorted, and have `.bai` indices.

In [ ]:
# ── Step 1: Index BAMs ────────────────────────────────────────────────────────
for sample, bam in BAMS.items():
    bai = Path(str(bam) + ".bai")
    if not bai.exists():
        print(f"Indexing {bam.name} …")
        subprocess.run(["samtools", "index", str(bam)], check=True)
    else:
        print(f"  {bam.name}: index already exists")

# Show sort order from BAM headers
for sample, bam in BAMS.items():
    result = subprocess.run(
        ["samtools", "view", "-H", str(bam)],
        capture_output=True, text=True
    )
    hd = [l for l in result.stdout.splitlines() if l.startswith("@HD")]
    pg = [l for l in result.stdout.splitlines() if l.startswith("@PG") and "minimap2" in l]
    print(f"\n{sample}:")
    if hd: print("  ", hd[0])
    if pg: print("  ", pg[0][:120])

In [ ]:
# ── Step 2: Run Clair3 on one sample (prototype on SBC4) ─────────────────────
# Runs run_clair3.sh — must have Clair3 installed (conda env sbi).
# Adjust CLAIR3_MODEL to your basecalling model before running.

PROTO_SAMPLE = "SBC4"
proto_bam    = BAMS[PROTO_SAMPLE]
proto_outdir = RESULTS_DIR / PROTO_SAMPLE
proto_outdir.mkdir(parents=True, exist_ok=True)

clair3_cmd = [
    "run_clair3.sh",
    f"--bam_fn={proto_bam}",
    f"--ref_fn={REF}",
    "--threads=8",
    "--platform=ont",
    f"--model_path={CLAIR3_MODEL}",
    f"--output={proto_outdir}",
    f"--sample_name={PROTO_SAMPLE}",
    "--include_all_ctgs",
]

print("Command:\n  " + " ".join(clair3_cmd))
print("\nRunning Clair3 … (this may take several minutes)")
result = subprocess.run(clair3_cmd, capture_output=True, text=True)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
else:
    print("Clair3 finished successfully.")
    vcf = proto_outdir / "merge_output.vcf.gz"
    print("Output VCF:", vcf, "— exists:", vcf.exists())

In [ ]:
# ── Step 3: Filter VCF (QUAL ≥ 20, DP ≥ 5, PASS only) ───────────────────────
MIN_QUAL  = 20
MIN_DEPTH = 5

raw_vcf      = proto_outdir / "merge_output.vcf.gz"
filtered_vcf = proto_outdir / f"{PROTO_SAMPLE}_filtered.vcf.gz"

filter_cmd = (
    f"bcftools filter "
    f"-e 'QUAL<{MIN_QUAL} || FORMAT/DP<{MIN_DEPTH}' {raw_vcf} "
    f"| bcftools view -f PASS -O z -o {filtered_vcf} "
    f"&& bcftools index --tbi {filtered_vcf}"
)
print("Filter command:", filter_cmd)
result = subprocess.run(filter_cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    print("Filtering done. Filtered VCF:", filtered_vcf)

In [ ]:
# ── Step 4: Summary stats on filtered VCF ────────────────────────────────────
import pandas as pd

stats_result = subprocess.run(
    ["bcftools", "stats", str(filtered_vcf)],
    capture_output=True, text=True
)

# Parse SN (summary numbers) section
rows = []
for line in stats_result.stdout.splitlines():
    if line.startswith("SN"):
        _, _, key, val = line.split("\t")[:4]
        rows.append({"Metric": key.strip(": "), "Value": int(val)})

df_stats = pd.DataFrame(rows)
print(f"\n=== {PROTO_SAMPLE} variant stats ===")
print(df_stats.to_string(index=False))